In [4]:
import time

In [5]:
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from lerobot.robots.so_follower import SO101Follower, SO101FollowerConfig
from lerobot.teleoperators.so_leader import SO101Leader, SO101LeaderConfig

from lerobot.processor import make_default_processors

In [6]:
dataset = LeRobotDataset("lerobot/pusht")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
teleop_action_processor, robot_action_processor, robot_observation_processor = make_default_processors()

In [8]:
robot_config = SO101FollowerConfig(
    port="/dev/ttyACM0",
    id="lhwdev_follower3",
)

leader_config = SO101LeaderConfig(
    port="/dev/ttyACM1",
    id="lhwdev_leader",
)

robot = SO101Follower(robot_config)
leader = SO101Leader(leader_config)

robot.connect()
leader.connect()

In [9]:
import cv2

cam = cv2.VideoCapture(2)

In [10]:
from IPython.display import display, Image
import cv2

_text_handle = None
_img_handle = None

def show_image(fps_text=None):
    global _text_handle, _img_handle
    
    ret, frame = cam.read()
    if not ret:
        return
        
    _, jpeg = cv2.imencode('.jpg', frame)
    jpeg_bytes = jpeg.tobytes()
    img_obj = Image(data=jpeg_bytes)
    
    if fps_text is not None:
        if _text_handle is None:
            _text_handle = display(fps_text, display_id=True)
        else:
            _text_handle.update(fps_text)
            
    if _img_handle is None:
        _img_handle = display(img_obj, display_id=True)
    else:
        _img_handle.update(img_obj)


In [11]:
def sync_to_robot():
    obs = robot.get_observation()
    raw_action = leader.get_action()
    teleop_action = teleop_action_processor((raw_action, obs))
    robot_action = robot_action_processor((teleop_action, obs))

    robot.send_action(robot_action)

In [12]:
from IPython.display import display, clear_output

start_time = time.perf_counter()
iterations = 0

# Reset display handles so a new display is created in this cell's output
_text_handle = None
_img_handle = None

while False:
    sync_to_robot()
    iterations += 1

    if iterations % 100 == 0:
        elipsed = time.perf_counter() - start_time
        fps_text = f"fps={iterations / elipsed:.3f} elipsed={elipsed:.2f}"
        show_image(fps_text)


In [14]:
import importlib
from record_interactive import record_interactive

record_interactive(dataset, robot, leader, cam)

HTML(value="\n    <style>\n    .studio-header {\n        background: linear-gradient(135deg, hsl(260, 80%, 40%…

UnboundLocalError: cannot access local variable 'recording_state' where it is not associated with a value